# Week 3 Discussion | Anatomy of a Failing Prompt
COSC 650 Applied LLM Systems

Ben Miller

## Discussion Post Prompt
Find or write a prompt that fails in a specific, reproducible way on a task you care about. It might be a prompt that drifts after several turns, a prompt that works on short inputs but collapses on long ones, or a prompt that produces correct answers for common cases but breaks on edge cases. Document the failure, then apply a structural fix using one of the anatomical components or patterns from this week.

In your post, address:

    What was the task, and what prompt did you start with?
    What exactly failed, and on what kind of input?
    Which failure mode category does it match (instruction drift, context contamination, format collapse, or something else)?
    What structural change did you make, and why did you expect it to help?
    Did the fix work? If not, what did you learn from the attempt?


In [1]:
import os
from openai import OpenAI
from google.colab import userdata
openai_api_key = userdata.get('OPENAI_API_KEY') or os.getenv("OPENAI_API_KEY")

if openai_api_key is None:
    raise ValueError("OPENAI_API_KEY not found. Please set it in Colab secrets or as an environment variable.")

openai_client = OpenAI(api_key=openai_api_key)

In [2]:
PROMPT = """Analyze the results for all of Roger Gracie's IBJJF matches at black belt.
Respond in one sentence that contains how many matches he won by submission,
which submission he used the most, and how many matches did he win with the
most popular submission."""

MESSAGES = [{"role": "user", "content": PROMPT}]

gpt_56_run1 = openai_client.chat.completions.create(
    model="gpt-5.6-sol",
    messages=MESSAGES
)

gpt_56_run2 = openai_client.chat.completions.create(
    model="gpt-5.6-sol",
    messages=MESSAGES
)

gpt_51_run1 = openai_client.chat.completions.create(
    model="gpt-5.1",
    messages=MESSAGES
)

gpt_51_run2 = openai_client.chat.completions.create(
    model="gpt-5.1",
    messages=MESSAGES
)

print("GPT 5.6 Sol Response 1: ", gpt_56_run1.choices[0].message.content)
print("GPT 5.6 Sol Response 2: ", gpt_56_run2.choices[0].message.content)
print()
print("GPT 5.1 Response 1: ", gpt_51_run1.choices[0].message.content)
print("GPT 5.1 Response 2: ", gpt_51_run2.choices[0].message.content)


GPT 5.6 Sol Response 1:  Roger Gracie won 48 black-belt IBJJF matches by submission, most frequently using the cross-collar choke, which earned him 18 victories.
GPT 5.6 Sol Response 2:  Roger Gracie won 54 black-belt IBJJF matches by submission, most often using the cross-collar choke from mount, which accounted for 23 wins.

GPT 5.1 Response 1:  Across all of Roger Gracie’s recorded IBJJF black belt matches, he won 20 by submission, his most used finish was the cross choke from mount, and he won 11 matches with that submission.
GPT 5.1 Response 2:  Across his IBJJF black belt career (Worlds, Pans, Europeans, Brazilian Nationals, ADCC trials-style IBJJF events excluded), Roger Gracie won 32 IBJJF black belt gi matches by submission, his most-used finishing technique was the cross choke from mount, and he secured that submission in 17 of those wins.


In [3]:
IMPROVED_PROMPT = """Analyze Roger Gracie's IBJJF competition record at black belt.

Include all IBJJF events, Gi and No-Gi, all weight divisions, and absolute divisions. Exclude ADCC, MMA, and non-IBJJF tournaments.

Perform the task in this order:

1. Identify the IBJJF events in which Roger Gracie competed at black belt.
2. Identify his matches from those events.
3. Separate submission wins from points, decisions, and other results.
4. Group equivalent names for the same submission technique together, including cross choke, cross-collar choke, and mounted cross-collar choke.
5. Count the submission wins and determine the most frequently used submission.

Respond in exactly one sentence stating how many IBJJF black-belt matches Roger Gracie won by submission, which submission he used most often, and how many wins came from that submission.

 """
MESSAGES_V2 = [{"role": "user", "content": IMPROVED_PROMPT}]

gpt_56_run1 = openai_client.chat.completions.create(
    model="gpt-5.6-sol",
    messages=MESSAGES_V2
)

gpt_56_run2 = openai_client.chat.completions.create(
    model="gpt-5.6-sol",
    messages=MESSAGES_V2
)

gpt_51_run1 = openai_client.chat.completions.create(
    model="gpt-5.1",
    messages=MESSAGES_V2
)
gpt_51_run2 = openai_client.chat.completions.create(
    model="gpt-5.1",
    messages=MESSAGES_V2
)

print("GPT 5.6 Sol Response 1: ", gpt_56_run1.choices[0].message.content)
print("GPT 5.6 Sol Response 2: ", gpt_56_run2.choices[0].message.content)
print()
print("GPT 5.1 Response 1: ", gpt_51_run1.choices[0].message.content)
print("GPT 5.1 Response 2: ", gpt_51_run2.choices[0].message.content)

GPT 5.6 Sol Response 1:  Roger Gracie won 48 IBJJF black-belt matches by submission, most frequently using the cross-collar choke, which accounted for 24 wins.
GPT 5.6 Sol Response 2:  Roger Gracie won 58 IBJJF black-belt matches by submission, most often using the cross-collar choke, which accounted for 24 wins.

GPT 5.1 Response 1:  Roger Gracie won 44 IBJJF black-belt matches by submission, with the cross-collar choke (including all equivalent names) as his most frequently used submission, accounting for 19 of those wins.
GPT 5.1 Response 2:  Roger Gracie won 38 IBJJF black-belt matches by submission, most often using the cross-collar choke from mount (including equivalent naming variants), which accounted for 21 of those wins.
